In [4]:
import numpy as np
import math, random
import pandas as pd
from scipy.spatial.transform import Rotation as rotate
from scipy.stats import truncnorm

from functions.representation import find_center_point_LWLC, load_objects
from functions.vectors import cosine_similarity, find_axis_of_rotation_geon_only, find_axis_of_rotation_geon_and_spatcon, same_object, calculate_axis_difficulty
from functions.plotting import add_frame, prepare_rotation_graphs

import plotly.graph_objects as go
import plotly.offline as pyo
import copy

# Initialize Plotly for offline mode in Jupyter Notebook
pyo.init_notebook_mode(connected=True)

# functions

def normal_dist_w_limits(low, high, mean, std):
    a = (low - mean) / std
    b = (high - mean) / std

    return truncnorm.rvs(a, b, loc=mean, scale=std)

def calculate_step_size_simple(angular_disparity, speed_constant):
    step_size = speed_constant * math.sqrt(angular_disparity)

    return step_size

# Experiment Variables

In [5]:
object_file = "JostJansenShapes.txt"
rotation_list = [0, 50, 100, 150]
# rotation_list = [0]
mirrored_list = [True, False]
# mirrored_list = [True]
axis_list = [np.array([0, 1, 0]), np.array([0, 0, 1])]
gender_list = [0, 1]                        # 0 = f, 1 = m
objects = load_objects(object_file)
n = 20                                      # number of subjects
average_run_time = 0
false_positives = 0
false_negatives = 0
m_counter = 0
f_counter = 0

csv_data = {
    "subject_num": [],
    "gender": [],
    "angle": [],
    "axis": [],
    "expected_answer": [],
    "axis_difficulty": [],
    "runtime": [],
    "decision": [],
    "num_of_repeats": [],
    "encoding_time": [],
    "landmarking_time": [],
    "rotation_time": [],
    "decision_time": []
}

# Main Loop

In [6]:
# ----------------- #
#   SUBJECT LOOP    #
# ----------------- #

subject_num = 0
for subject in range(n):

    # SUBJECT VARIABLES:

    subject_num += 1
    # gender = random.choice(gender_list)
    # if gender == 0:
    #     f_counter += 1
    # else:
    #     m_counter += 1

    if subject_num <= n/2:
        gender = 0
        f_counter += 1
    else:
        gender = 1
        m_counter += 1

    # timing
    production_time = 50                                                                            # add distribution?
    propositional_difficulty_time = None                                                            # determined later
    object_encoding_time = normal_dist_w_limits(low=150, high=300, mean=225, std=25)                # about 150-300ms for each object. TBH look up what mean and std are for object encoding if possible

    # rotation
    speed_constant = None                                                                           # determined later
    speed_constant_decrease = 0.5                                                                   # add distribution?

    # decision
    repeat_threshold = round(normal_dist_w_limits(low=1, high=5, mean=2 + (1-gender), std=1))       # 1-5 repeated steps
    wrong_guess_decrease = 0.1                                                                     # add distribution affected by gender/axis difficulty?

    # similarity thresholds
    geon_alignment_threshold = normal_dist_w_limits(low=0.6, high=1, mean=0.8, std=0.05)            # cosine similarity threshold for landmark geon alignment before next step starts--> maybe 0.7 - 0.9
    landmark_angle_threshold = normal_dist_w_limits(low=0.9, high=1, mean=0.98, std=0.01)           # cosine similarity threshold for landmark geon and spat con alignment during 2nd step
    object_angle_threshold = normal_dist_w_limits(low=0.9, high=1, mean=0.95, std=0.02)

    # --------------------- #
    #   TEST OBJECT LOOP    #
    # --------------------- #

    for object in objects:

        # ROTATION VARIABLES:

        total_run_time = 0
        encoding_time = 0
        landmarking_time = 0
        rotation_time = 0
        decision_time = 0
        wrong_guess_chance = 0.5                                         # starts at 50%
        angle = random.choice(rotation_list)
        axis = random.choice(axis_list)
        mirrored = random.choice(mirrored_list)
        axis_difficulty = calculate_axis_difficulty(axis)
        r = rotate.from_rotvec(np.deg2rad(angle) * axis)
        center_point = np.array([0,0,0])

        # REMAINING SUBJECT VARIABLES (affected by axis difficulty):

        propositional_difficulty_time = normal_dist_w_limits(low=1, high=3, mean=1.5 + (0.5 * axis_difficulty), std=1)              # 1-3 ms --> more difficult on combination rotations
        speed_constant = normal_dist_w_limits(low=1, high=5, mean=3 + (0.5 * gender) - axis_difficulty, std=2)                      # idek like 2-4 seems realistic? maybe reduce on repeat --> slower start on combination rotations

        # MAKE ORIGINAL AND TARGET OBJECTS:

        original_object = copy.deepcopy(object)
        target_object = copy.deepcopy(object)
        target_object.rotate(r)
        if mirrored:
            target_object.flip_last_geon()

        # make copy of original object
        original_object_reset = copy.deepcopy(original_object)

        # CALCULATE ROTATION DATA:
        axis_of_rotation, direction, curr_step_angular_disparity = find_axis_of_rotation_geon_only(original_object, target_object, center_coords=center_point)
        total_axis, _, total_angular_disparity = find_axis_of_rotation_geon_and_spatcon(original_object, target_object, center_coords=center_point)

        # TIMING:

        # add time needed for representation
        encoding_time += 2 * object_encoding_time
        total_run_time = total_run_time + encoding_time
        
        # add time needed for landmarking
        total_run_time = total_run_time + production_time                                                                 # check geon
        total_run_time = total_run_time + production_time + (total_angular_disparity * propositional_difficulty_time)     # check spatial connection
        landmarking_time += production_time
        landmarking_time += production_time + (total_angular_disparity * propositional_difficulty_time)

        # ----------------- #
        #   ROTATON LOOP    #
        # ----------------- #

        repeat_count = 0
        same = False
        decision = None
        while not same and repeat_count < repeat_threshold:

            # prepare graphs
            # axis_animation_fig, overlap_animation_fig, sidebyside_animation_fig = prepare_rotation_graphs(original_object, target_object, axis_of_rotation, center_point, production_time)
            # axis_animation = []
            # overlap_animation = []
            # sidebyside_animation = []

            # set landmark vectors, angular disparity
            original_landmark_geon_vector = original_object.get_landmark_geon().get_vector()
            target_landmark_geon_vector = target_object.get_landmark_geon().get_vector()
            original_spatcon_direction = original_object.get_landmark_spatial_connection().get_vector()
            target_spatcon_direction = target_object.get_landmark_spatial_connection().get_vector()
            total_angular_disparity_part2 = None

            # STEP ONE: GEON ALIGNMENT

            loop_count = 0
            while cosine_similarity(original_landmark_geon_vector, target_landmark_geon_vector) < geon_alignment_threshold:     # checking cosine similarity between target and goal geons

                # find best axis/direction of rotation, angular disparity
                axis_of_rotation, direction, curr_step_angular_disparity = find_axis_of_rotation_geon_only(original_object, target_object, center_coords=center_point, prev_axis=axis_of_rotation, prev_direction=direction, prev_angle=curr_step_angular_disparity, total_angular_disparity=total_angular_disparity)

                # calculate step size based on angular disparity and rotation speed
                step_size = calculate_step_size_simple(curr_step_angular_disparity, speed_constant)
                print("CURR_ANGULAR_DISPARITY: " + str(curr_step_angular_disparity) + ", STEP_SIZE: " + str(step_size))

                # apply rotation to original object
                r = rotate.from_rotvec(direction * np.deg2rad(step_size) * axis_of_rotation)        # quaternion representing step size rotation around calculated axis
                original_object.rotate(r)

                # update original geon and spatial connection vectors
                original_landmark_geon_vector = original_object.get_landmark_geon().get_vector()
                original_spatcon_direction = original_object.get_landmark_spatial_connection().get_vector()

                # add animation frames to graphs
                # add_frame(axis_animation, original_object, object_name="Original Object", landmark_name="Original Landmarks", axis_of_rotation=axis_of_rotation, object_colour='blue', landmark_colour='purple', axis_colour='green', axis_scale=2)
                # original_centerpoint_vec = find_center_point_LWLC(original_object)
                # copy_og_obj = copy.deepcopy(original_object)
                # copy_og_obj.update_start_coords(copy_og_obj.start_coords - original_centerpoint_vec)
                # add_frame(overlap_animation, copy_og_obj, object_name="Original Object", object_colour='blue', axis_scale=2)
                # add_frame(sidebyside_animation, copy_og_obj, object_name="Original Object", object_colour='blue', axis_scale=2)

                loop_count += 1
                total_run_time += production_time * 3
                rotation_time += production_time * 3

                # emergency break
                if loop_count > 100:
                    break

            # STEP TWO: GEON AND SPAT CON ALIGNMENT:

            loop_count = 0
            while cosine_similarity(original_landmark_geon_vector, target_landmark_geon_vector) < landmark_angle_threshold or cosine_similarity(original_spatcon_direction, target_spatcon_direction) < landmark_angle_threshold:     # checking cosine similarity between target and goal geons and spatial connections

                # find best axis/direction of rotation, angular disparity
                axis_of_rotation, direction, curr_step_angular_disparity = find_axis_of_rotation_geon_and_spatcon(original_object, target_object, center_coords=center_point, prev_axis=axis_of_rotation, prev_direction=direction, prev_angle=curr_step_angular_disparity, total_angular_disparity=total_angular_disparity)

                # calculate step size based on angular disparity and rotation speed
                if total_angular_disparity_part2 == None:
                    total_angular_disparity_part2 = curr_step_angular_disparity
                step_size = calculate_step_size_simple(curr_step_angular_disparity, speed_constant)
                print("CURR_ANGULAR_DISPARITY: " + str(curr_step_angular_disparity) + ", STEP_SIZE: " + str(step_size))

                # apply rotation to original object
                r = rotate.from_rotvec(direction * np.deg2rad(step_size) * axis_of_rotation)        # quaternion representing step size rotation around calculated axis
                original_object.rotate(r)

                # update original geon and spatial connection vectors
                original_landmark_geon_vector = original_object.get_landmark_geon().get_vector()
                original_spatcon_direction = original_object.get_landmark_spatial_connection().get_vector()

                # add animation frames to graphs
                # add_frame(axis_animation, original_object, object_name="Original Object", landmark_name="Original Landmarks", axis_of_rotation=axis_of_rotation, object_colour='blue', landmark_colour='purple', axis_colour='green', axis_scale=2)
                # original_centerpoint_vec = find_center_point_LWLC(original_object)
                # copy_og_obj = copy.deepcopy(original_object)
                # copy_og_obj.update_start_coords(copy_og_obj.start_coords - original_centerpoint_vec)
                # add_frame(overlap_animation, copy_og_obj, object_name="Original Object", object_colour='blue', axis_scale=2)
                # add_frame(sidebyside_animation, copy_og_obj, object_name="Original Object", object_colour='blue', axis_scale=2)

                loop_count += 1
                total_run_time += production_time * 3
                rotation_time += production_time * 3

                # emergency break
                if loop_count > 100:
                    break

            # axis_animation_fig.frames = axis_animation
            # axis_animation_fig.show()
            # overlap_animation_fig.frames = overlap_animation
            # overlap_animation_fig.show()
            # sidebyside_animation_fig.frames = sidebyside_animation
            # sidebyside_animation_fig.show()

            # check overall similarity

            same, similarity_check_run_time = same_object(original_object, target_object, object_angle_threshold, total_angular_disparity, production_time, propositional_difficulty_time)
            total_run_time += similarity_check_run_time
            decision_time += similarity_check_run_time

            if not same:

                # reset original object
                original_object = original_object_reset
                axis_of_rotation, direction, total_angular_disparity = find_axis_of_rotation_geon_only(original_object, target_object, center_coords=center_point)
                speed_constant = speed_constant - speed_constant_decrease
                wrong_guess_chance = wrong_guess_chance - wrong_guess_decrease
                repeat_count += 1

                print("REPEAT #" + str(repeat_count))
                print("\nRUN TIME:\n" + str(round(total_run_time/1000, 3)) + " seconds")
                print("speed_constant: " + str(speed_constant))

        print("DECISION:")
        if same:
            print("same")
            decision = "same"
        else:
            if random.uniform(0, 1) <= (total_angular_disparity/180) * wrong_guess_chance:              # chance that wrong guess occurs
                print("same")
                decision = "same"
            else:
                print("different")
                decision = "different"

        print()
        print("gender: f") if gender == 0 else print("gender: m")

        print()
        print("propositional_difficulty_time: " + str(propositional_difficulty_time))
        print("object_encoding_time: " + str(object_encoding_time))
        print("speed_constant: " + str(speed_constant))
        print("repeat_threshold: " + str(repeat_threshold))
        print("geon_alignment_threshold: " + str(geon_alignment_threshold))
        print("landmark_angle_threshold: " + str(landmark_angle_threshold))
        print("object_angle_threshold: " + str(object_angle_threshold))

        print("angle: " + str(angle))
        print("axis: " + str(axis))
        print("mirrored: " + str(mirrored))
        print("axis_difficulty: " + str(axis_difficulty))
        print("\nRUN TIME:\n" + str(round(total_run_time/1000, 3)) + " seconds")
        print("decision: " + decision)
        print("number of repeats: " + str(repeat_count))

        csv_data['subject_num'].append(subject_num)
        csv_data['gender'].append(gender)
        csv_data['angle'].append(angle)
        csv_data['axis'].append(axis)
        csv_data['expected_answer'].append("different" if mirrored else "same")
        csv_data['axis_difficulty'].append(axis_difficulty)
        csv_data['runtime'].append(round(total_run_time/1000, 3))
        csv_data['decision'].append(decision)
        csv_data['num_of_repeats'].append(repeat_count)
        csv_data['encoding_time'].append(round(encoding_time/1000, 3))
        csv_data['landmarking_time'].append(round(landmarking_time/1000, 3))
        csv_data['rotation_time'].append(round(rotation_time/1000, 3))
        csv_data['decision_time'].append(round(decision_time/1000, 3))

        if mirrored and decision == "same":
            false_positives += 1
        elif not mirrored and decision == "different":
            false_negatives += 1

        average_run_time += round(total_run_time/1000, 3)

print("total runs: " + str(n*len(objects)))
print("average_run_time: " + str(average_run_time/(n*len(objects))))
print("false_positives: " + str(false_positives))
print("false_negatives: " + str(false_negatives))
print("num of boys: " + str(m_counter))
print("num of girls: " + str(f_counter))

# save experiment data to csv
df = pd.DataFrame(csv_data)
df.to_csv('rotation_model_data.csv', index=False)



CURR_ANGULAR_DISPARITY: 99.85107611658391, STEP_SIZE: 17.267302732836363
CURR_ANGULAR_DISPARITY: 82.79046028576165, STEP_SIZE: 15.72310363187203
CURR_ANGULAR_DISPARITY: 68.66552719159156, STEP_SIZE: 14.319158874646936
CURR_ANGULAR_DISPARITY: 58.77905850543278, STEP_SIZE: 13.248278254882042
CURR_ANGULAR_DISPARITY: 52.32229873668318, STEP_SIZE: 12.499468211225699
CURR_ANGULAR_DISPARITY: 26.9426882945369, STEP_SIZE: 8.969507382914191
CURR_ANGULAR_DISPARITY: 17.97318091162269, STEP_SIZE: 7.325893506633279
REPEAT #1

RUN TIME:
2.633 seconds
speed_constant: 1.2280174680570737
CURR_ANGULAR_DISPARITY: 99.85107611658391, STEP_SIZE: 12.271027217100068
CURR_ANGULAR_DISPARITY: 87.73075467113843, STEP_SIZE: 11.502188407601805
CURR_ANGULAR_DISPARITY: 76.60835831027808, STEP_SIZE: 10.748370352696107
CURR_ANGULAR_DISPARITY: 67.45978853175684, STEP_SIZE: 10.08618738172281
CURR_ANGULAR_DISPARITY: 60.40535639781839, STEP_SIZE: 9.544260184212646
CURR_ANGULAR_DISPARITY: 55.140293307901985, STEP_SIZE: 9.118